In [1]:
import numpy as np

import numpy as np

from matplotlib import pyplot as plt

def init_random_vec(d):
    a_vec = ((np.random.rand(d) * 2) - 1) * np.pi
    c_vec = np.exp(a_vec*1j)
    return c_vec

def init_random_mat(d1, d2):
    a_vec = ((np.random.rand(d1, d2) * 2) - 1) * np.pi
    c_vec = np.exp(a_vec*1j)
    return c_vec

def forward_opp(mat1, mat2):
    d = mat1.shape[0]
    return np.matmul(mat1, mat2)/d
def s_opp(mat1, mat2):
    d = mat1.shape[0]
    return np.matmul(mat1, np.transpose(np.conj(mat2)))/d

def similarity(vec1, vec2):
    d = vec1.shape[0]
    return np.real(np.dot(vec1, np.conj(vec2))/d)

def bind(vec1, vec2):
    return vec1 * vec2

def inverse(vec):
    return normalize(np.conj(vec)) / np.absolute(vec)

def unbind(vec1, vec2):
    return bind(vec1, inverse(vec2))

def bundle(vec1, *args):
    new_vec = vec1
    for vec in args:
        new_vec = new_vec + vec
    #new_vec = new_vec / (len(args)+1)
    return new_vec
    
def normalize(vec):
    a_vec = np.angle(vec)
    return np.exp(a_vec*1j)

def frac_power(c_vec, x):
    a_vec = np.angle(c_vec)
    a_vec = a_vec * x
    c_vec = np.exp(a_vec*1j)
    return c_vec

In [2]:
test_v = init_random_vec(10)
test_mat = init_random_mat(10, 10)
print(forward_opp(test_mat, test_v))

[-0.17845479-0.02331518j  0.17124874-0.21761255j -0.01859941-0.34477107j
  0.07218006-0.48902936j -0.2643366 -0.01426714j -0.08593392-0.12255106j
 -0.28190487-0.13043766j -0.26215236+0.11380385j  0.28835488+0.22576599j
 -0.24994132-0.36013621j]


In [3]:
in_dim = 1000
out_dim = 1000

In [4]:
zero_zero_vec = init_random_vec(in_dim)
zero_one_vec = init_random_vec(in_dim)
one_zero_vec = init_random_vec(in_dim)
one_one_vec = init_random_vec(in_dim)


def encode(x, y):
    if x == 0 and y == 0:
        return zero_zero_vec
    elif x == 0 and y == 1:
        return zero_one_vec
    elif x == 1 and y == 0:
        return one_zero_vec
    else:
        return one_one_vec

In [5]:
out_zero_vec = init_random_vec(out_dim)
out_one_vec = init_random_vec(out_dim)
def reward_f(vec, x):
    if x == 1:
        return similarity(vec, out_one_vec)
    else:
        return similarity(vec, out_zero_vec)
def decode(x):
    if x == 1:
        return out_one_vec
    else:
        return out_zero_vec

In [6]:
import random

In [7]:
x_train_val = [random.choices([0, 1], k=2) for _ in range(100)]
y_train = [(x[0] + x[1]) % 2 for x in x_train_val]
x_train = [encode(x[0], x[1]) for x in x_train_val]

In [8]:
x_test_val = [random.choices([0, 1], k=2) for _ in range(10)]
y_test = [(x[0] + x[1]) % 2 for x in x_test_val]
x_test = [encode(x[0], x[1]) for x in x_test_val]

In [40]:
class network:
    def __init__(self, in_dim, out_dim, middle_dim):
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.middle_dim = middle_dim
        self.layer_1 = init_random_mat(middle_dim, in_dim)
        self.layer_2 = init_random_mat(out_dim, middle_dim)
        self.learn_rate = 0.1

    def train(self, x, y, lr, noise = 0.1):
        middle = normalize(forward_opp(self.layer_1, x) + (init_random_mat(self.middle_dim, x.shape[1])*noise))
        out = normalize(forward_opp(self.layer_2, middle) + (init_random_mat(self.middle_dim, x.shape[1])*noise))
        reward = np.clip(np.real(np.sum(out * np.transpose(np.array([decode(label) for label in y])), axis=0)/out.shape[0]), 0, None)
        #max_reward = np.max(reward)
        #reward_multiplier = lr/max_reward
        #scaled_reward = reward * reward_multiplier
        sum_reward = np.sum(np.abs(reward))
        summed_change_matrix_1 = np.zeros_like(self.layer_1)
        summed_change_matrix_2 = np.zeros_like(self.layer_2)
        for i in range(x.shape[1]):
            change_matrix_1 = np.matmul(middle[:, i:i+1],np.transpose(np.conj(x[:, i:i+1])))*reward[i]
            summed_change_matrix_1 += change_matrix_1
            
            change_matrix_2 = np.matmul(out[:, i:i+1],np.transpose(np.conj(middle[:, i:i+1])))*reward[i]
            summed_change_matrix_2 += change_matrix_2
        self.layer_1 = normalize(self.layer_1 + summed_change_matrix_1*lr/sum_reward)
        self.layer_2 = normalize(self.layer_2 + summed_change_matrix_2*lr/sum_reward)
        return reward
    def eval(self, x, y):
        middle = normalize(forward_opp(self.layer_1, x))
        out = normalize(forward_opp(self.layer_2, middle))
        reward = np.real(np.sum(out * np.transpose(np.array([decode(label) for label in y])), axis=0)/out.shape[0])
        return reward


In [50]:
test_network = network(1000, 1000, 1000)

In [51]:

print(f'starting_reward = {np.mean(test_network.eval(np.transpose(np.array(x_test)), y_test))}')
for step in range(100):
    print(f'step {step} -----------------')
    for i in range(len(x_train)):
        test_network.train(np.transpose(np.array([x_train[i] for _ in range(10)])), [y_train[i] for _ in range(10)], 0.01, 0.2)
    print(f'test_reward = {np.mean(test_network.eval(np.transpose(np.array(x_test)), y_test))}')

starting_reward = 0.0022575534060126673
step 0 -----------------
test_reward = 0.015048246116499187
step 1 -----------------
test_reward = 0.04356152099185199
step 2 -----------------
test_reward = 0.06006324414512032
step 3 -----------------
test_reward = 0.08156844936362095
step 4 -----------------
test_reward = 0.11331636683677833
step 5 -----------------
test_reward = 0.14423089661614902
step 6 -----------------
test_reward = 0.17287766883954112
step 7 -----------------
test_reward = 0.18496576779916835
step 8 -----------------
test_reward = 0.1939460710209173
step 9 -----------------
test_reward = 0.18930176602766224
step 10 -----------------
test_reward = 0.1799833097556714
step 11 -----------------
test_reward = 0.16753601201379612
step 12 -----------------
test_reward = 0.15563391190399034
step 13 -----------------
test_reward = 0.14692930158725803
step 14 -----------------
test_reward = 0.13900153769554174
step 15 -----------------
test_reward = 0.1335220644185498
step 16 ----

KeyboardInterrupt: 

In [52]:
print(test_network.eval(np.transpose(np.array([encode(0, 0)])), [0]))
print(test_network.eval(np.transpose(np.array([encode(0, 0)])), [1]))
print(test_network.eval(np.transpose(np.array([encode(0, 1)])), [0]))
print(test_network.eval(np.transpose(np.array([encode(0, 1)])), [1]))
print(test_network.eval(np.transpose(np.array([encode(1, 0)])), [0]))
print(test_network.eval(np.transpose(np.array([encode(1, 0)])), [1]))
print(test_network.eval(np.transpose(np.array([encode(1, 1)])), [0]))
print(test_network.eval(np.transpose(np.array([encode(1, 1)])), [1]))

[0.06433326]
[-0.03795792]
[0.21537202]
[0.01589932]
[-0.23122202]
[0.01110986]
[0.23848123]
[0.00372766]
